In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from ast import literal_eval
import pandas as pd
import numpy as np

csv_file_path2 = '/content/drive/MyDrive/project/data/new_datasettt.csv'  # Replace with your desired file path

# Read CSV file back into DataFrame
df_read = pd.read_csv(csv_file_path2)


# Convert string representations back to lists
df_read['I'] = df_read['I'].apply(lambda x: list(map(float, x.split(','))))
df_read['Q'] = df_read['Q'].apply(lambda x: list(map(float, x.split(','))))
df_read['I & Q'] = df_read['I & Q'].apply(lambda x: list(map(float, x.split(','))))
df_read['frequency_offset'] = df_read['frequency_offset'].apply(literal_eval).apply(lambda x: x[0])
df_read['snr'] = df_read['snr'].apply(literal_eval).apply(lambda x: x[0])




In [ ]:
def combine_info(row):
    return (row['I & Q'], row['modulation_type'], row['snr'],row['frequency_offset'])

# Apply the function to each row to create the new column
df_read['combined_info'] = df_read.apply(combine_info, axis=1)


# Print the DataFrame to verify the new column



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Assuming df_read is your DataFrame and combined_info is already created
# Convert 'combined_info' to a DataFrame with individual columns
combined_df = pd.DataFrame(df_read['combined_info'].tolist(), columns=['I_Q', 'modulation_type', 'snr', 'frequency_offset'])

# Shuffle the DataFrame
combined_df = combined_df.sample(frac=1, random_state=60).reset_index(drop=True)

# Split the data into train, validation, and test sets
# First, split into train+val and test
train_val_df1, test_df = train_test_split(combined_df, test_size=0.2, stratify=combined_df[['modulation_type', 'snr']], random_state=70)

# Then, split the train+val into train and val
xtrain_df, xval_df = train_test_split(train_val_df1, test_size=0.25, stratify=train_val_df1[['modulation_type', 'snr']], random_state=69)





In [ ]:
combined_df

In [ ]:
import numpy as np

# Convert 'I_Q' column of xtrain_df to a NumPy array
xtrain_array = np.array(xtrain_df['I_Q'].tolist())

xval_array = np.array(xval_df['I_Q'].tolist())
xtest_array = np.array(test_df['I_Q'].tolist())

ytrain_array = np.array(xtrain_df['frequency_offset'].tolist())
yval_array = np.array(xval_df['frequency_offset'].tolist())
ytest_array = np.array(test_df['frequency_offset'].tolist())

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense,GRU
import tensorflow as tf
def create_fcn(input_dim):
    inputs = Input(shape=(input_dim,))
    hidden1 = Dense(500, activation='relu')(inputs)
    dropout1 = Dropout(0.2)(hidden1)
    hidden2 = Dense(600, activation='relu')(dropout1)
    dropout2 = Dropout(0.2)(hidden2)
    hidden3= Dense(300, activation='relu')(dropout2)
    dropout3 = Dropout(0.2)(hidden3)
    outputs = Dense(1, activation='linear')(dropout3)
    model = Model(inputs=inputs, outputs=outputs)
    return model

In [ ]:
from tensorflow.keras.layers import Dropout, BatchNormalization,Reshape
model = create_fcn(256)

In [ ]:
from keras.optimizers import AdamW
from keras.optimizers.schedules import ExponentialDecay
from keras.optimizers import Adam


model.compile(Adam(learning_rate=0.001), loss='mse', metrics=['mae'])


In [ ]:

from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

history = model.fit(xtrain_array, ytrain_array, epochs=500, batch_size=128, validation_data=(xval_array, yval_array),callbacks=[early_stop])


In [ ]:
train_loss = history.history['loss']
val_loss = history.history['val_loss']


In [ ]:
train_mae = history.history['mae']
val_mae = history.history['val_mae']

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_loss) + 1)

# Plotting loss
plt.figure(figsize=(6, 3))
plt.plot(epochs, train_loss, 'r', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Plotting MAE if available
if 'mae' in history.history:
    plt.figure(figsize=(6, 3))
    plt.plot(epochs, train_mae, 'r', label='Training MAE')
    plt.plot(epochs, val_mae, 'b', label='Validation MAE')
    plt.title('Training and validation MAE')
    plt.xlabel('Epochs')
    plt.ylabel('MAE')
    plt.legend()
    plt.grid(True)
    plt.show()
